# <h1 style="color:#9bddff">Portfolio Optimization & Risk Profile Analytics</h1>

In [1]:
%load_ext autoreload
%autoreload 2

> EQUITY8 Portfolios - Selection Widget<br>
> Task Selection (json stored tasks)<br>
> Populate Goals and Constraints for Selected Task<br>
> Define Min, Max, and Step for Constraints<br>
> Run Optimizations (Cache Results by portfolio task, constraints, as of date) - Define Task IDs for each task - even same task but with different constraint tolerances<br>

In [2]:
from bloomberg.enterprise.workflow import ApiClient
from bloomberg.enterprise.workflow.api import WorkflowsApi, WorkflowRunsApi, ExternalResourcesApi
from bloomberg.enterprise.workflow.models.workflow_for_create import WorkflowForCreate
from bloomberg.enterprise.oauth import OAuthFlow
from bloomberg.environments import ApiTier

from bloomberg.enterprise.portfolio.report import ApiClient as PEDLApiClient
from bloomberg.enterprise.portfolio.report.api import PortfolioDataApi, PortfolioReportApi
from bloomberg.enterprise.portfolio.report.models.portfolio_dataset import PortfolioDataset

import json
import pandas as pd
import numpy as np
from itables import show
from pathlib import Path
import yaml

import os
import re

#import cvxopt as opt
#from cvxopt import blas, solvers

import ipywidgets as ipw
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [3]:
import logging
import json
import bloomberg.enterprise.portfolio.optimization as SDK
from bloomberg.environments import ApiTier
from bloomberg.enterprise.oauth import OAuthFlow

In [4]:
from enums import TaskStatus
from portfolio import Portfolio
from optimization import Optimizer, OptimizationTask, OptimizationResult
from utils.logging_config import setup_logger

In [5]:
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

In [6]:
## Credentials

# # Set Proxy - if you do not have proxy, you may ignore this
# proxy_url = "http://bproxy.tdmz1.bloomberg.com:80"

config_path = "port_v2_config.json"

with open(config_path, "r") as config_file:
    raw_config = config_file.read()
config = json.loads(raw_config)

CLIENT_ID = config['client_id']
CLIENT_SECRET = config['client_secret']

In [8]:
# Create an API client session -- Add proxy_url = proxy_url if you have proxy in your firm.
api_client = ApiClient(CLIENT_ID, CLIENT_SECRET, OAuthFlow.CLIENT_CREDENTIALS)
pedl_api_client = PEDLApiClient(CLIENT_ID, CLIENT_SECRET, OAuthFlow.CLIENT_CREDENTIALS)

# Create a workflow API instance
api_workflows = WorkflowsApi(api_client)

In [9]:
# single_stock_limit = config_manager.get_component('instrument_constraints', 'single_stock_limit')
# task_config['instrument_constraints'] = [
#     #config_manager.get_component('instrument_constraints', 'single_stock_limit')
#     SDK.InstrumentConstraint(
#         scope = SDK.InstrumentConstraintScope(
#             SDK.SingleInstrumentScope(
#                 instrument_unique_id = single_stock_limit['scope']['instrumentUniqueId']
#             )
#         ),
#         units = single_stock_limit['units'],
#         fields = [
#             SDK.InstrumentConstraintFieldsInner(
#                 field_code = field['field_code'],
#                 value_or_field = SDK.InstrumentConstraintFieldsInnerValueOrField(
#                     SDK.InstrumentConstraintValue(value = field['value_or_field']['value']))
#             ) 
#             for field in single_stock_limit['fields']
#         ]
#     )
# ]

In [7]:
from config_manager import OptimizationConfigManager

config_manager = OptimizationConfigManager()

# print("Available optimization tasks:")
# print(config_manager.list_available_tasks())

In [11]:
tech_constraint = config_manager.get_component('instrument_constraints', 'single_stock_limit')
# config_manager.get_component('portfolio_constraints','turnover_5')
# config_manager.get_component('portfolio_constraints','tech_sector_limit')
# config_manager.get_component('portfolio_constraints','max_trades_20')

# config_manager.get_component('goals','maximize_expected_return')

In [12]:
max_pos_constraints = [
    'max_pos_500',
    #'max_pos_400',
    #'max_pos_150',
    #'max_pos_100',
    #'max_pos_75',
    #'max_pos_50'
]

turnover_constraints = [
    #'turnover_1',
    #'turnover_5',
    #'turnover_10',
    #'turnover_15',
    'turnover_20'
]
active_risk_constraints = [
    #'active_risk_015',
    #'active_risk_016',
    #'active_risk_017',
    #'active_risk_018',
    #'active_risk_019',
    'active_risk_02',
    #'active_risk_021',
    # 'active_risk_0225',
    #'active_risk_023',
    #'active_risk_024',
    # 'active_risk_025',
    # 'active_risk_026',
    # 'active_risk_0275',
    #'active_risk_028',
    #'active_risk_029',
    # 'active_risk_03',
]



# Initialize the SDK client
client = SDK.ApiClient(
    CLIENT_ID,
    CLIENT_SECRET,
    OAuthFlow.DEVICE_CODE  # This tells the SDK to use device code flow for authentication
)

# Create optimizer task from config
#portfolio = Portfolio(portfolio_id="ENTRY_EXIT_SAMPLE", name="Test Portfolio")
portfolio = Portfolio(portfolio_id="EQUITY8_US", name="Test Portfolio")
#portfolio = Portfolio(portfolio_id="CANADA FORUM DEMO 2024", name="Test Portfolio")

In [8]:
task_config = config_manager.get_task_config("balanced_strategy")
task_config['risk_options'] = SDK.OptimizationTaskRiskOptions(
    risk_model = SDK.RiskModelEnum.US_EQUITY,
    #risk_model_scaling = SDK.RiskModelScalingEnum.WEEK,
    risk_model_horizon = SDK.RiskModelHorizonEnum.ANNUAL
)
#task_config

In [9]:
task_config

{'benchmark_id': 'SPX',
 'description': 'Balancing risk minimization with return targets',
 'goals': [{'action': 'MAXIMIZE',
   'fieldCode': "CUSTOM_NUMBER(FIELD='CF_7459422178055815169')"}],
 'instrument_constraints': [],
 'name': 'Risk-Return Balanced Strategy',
 'portfolio_constraints': [{'fieldCode': 'turnover',
   'maxThreshold': 20,
   'relativeTo': None},
  {'aggregation': 'GROSS_VALUE',
   'classificationNode': {'classificationName': 'GICS Sector',
    'levels': ['Information Technology']},
   'fieldCode': 'weight',
   'maxThreshold': 0.5}],
 'trade_universe': [{'instrumentSource': {'id': 'SPX', 'type': 'INDEX_TICKER'},
   'tradingRule': 'BUY_AND_SELL',
   'useAsSecondaryBenchmark': False}],
 'risk_options': OptimizationTaskRiskOptions(risk_model=<RiskModelEnum.US_EQUITY: 'US_EQUITY'>, risk_model_scaling=None, risk_model_horizon=<RiskModelHorizonEnum.ANNUAL: 'ANNUAL'>)}

In [14]:
# results_data = []
# failed_tasks = []
# for pos in max_pos_constraints:
#     for turnover in turnover_constraints:
#         for risk in active_risk_constraints:
#             # Get the result for this combination
#             task_config['portfolio_constraints'] = [
#                 config_manager.get_component('portfolio_constraints', turnover),
#                 config_manager.get_component('portfolio_constraints', risk),
#                 config_manager.get_component('portfolio_constraints', pos)
#                 #config_manager.get_component('portfolio_constraints','max_trades_20'),
#             ]

            

#             task_id = portfolio.create_task(**task_config)
            
#             # Run optimization
#             optimizer = Optimizer(client, SDK)
#             result = portfolio.run_task(task_id, optimizer)
#             optimization_api = SDK.OptimizationsApi(base_api_client=client)
#             result.wait_for_completion(optimization_api, 200)
            
#             summary_df = result.get_summary_dataframe()
#             goals_frame = result.get_goals_dataframe()
#             constraint_frame = result.get_constraints_dataframe()
            
#             if not summary_df.empty and not goals_frame.empty and not constraint_frame.empty:
#                 results_data.append({
#                     'pos_constraint': pos,
#                     'turnover_constraint': turnover,
#                     'risk_constraint': risk,
#                     'actual_pos': constraint_frame.loc[constraint_frame['fieldCode']=='maximum_positions', 'finalValue'].values[0],
#                     'actual_turnover': constraint_frame.loc[constraint_frame['fieldCode']=='turnover', 'finalValue'].values[0],
#                     'actual_risk': constraint_frame.loc[constraint_frame['fieldCode']=='active_total_risk', 'finalValue'].values[0],
#                     'expected_return': goals_frame.iloc[0,2],
#                     'task_id': task_id
#                 })

#                 trades_frame = result.get_trades_dataframe()
#                 path = f"trades/trades_{pos}_{turnover}_{risk}_{task_id}.parquet.gzip"
#                 trades_frame.to_parquet(path, compression='gzip')
#             else:
#                 print(f"No optimization data for Task ID: {task_id}")
#                 failed_tasks.append(result)

# results_df = pd.DataFrame(results_data)

In [15]:
#results_df

In [15]:
optimization_api = SDK.OptimizationsApi(base_api_client=client)

In [16]:
from visualization.data_manager import OptimizationDataManager, OrderManager
from visualization.plot_manager import ParallelCoordinatesPlotter
from visualization.gui import PortfolioOptimizationGUI

import pandas as pd

In [17]:
path = 'optimization_results_pos_turnover_risk_er_20250114.parquet.gzip'
results_df = pd.read_parquet(path)

data_manager = OptimizationDataManager(results_df)
gui = PortfolioOptimizationGUI(data_manager)

display(gui.view)

In [20]:
# trades_frame = gui.data_manager.get_trades_for_task(gui.task_dd.value.split(' ')[0])
# trades_frame.head()

In [21]:
# from optimization_manager import OptimizationResultsManager
# from constraint_generator import ConstraintGenerator

In [22]:
# # Initialize the SDK client
# client = SDK.ApiClient(
#     CLIENT_ID,
#     CLIENT_SECRET,
#     OAuthFlow.DEVICE_CODE  # This tells the SDK to use device code flow for authentication
# )
# optimization_api = SDK.OptimizationsApi(base_api_client=client)

In [23]:
# # Initialize the results manager
# results_manager = OptimizationResultsManager()
# # Initialize constraint generator
# constraint_gen = ConstraintGenerator()

# optimizer = Optimizer(client, SDK)
# optimization_api = SDK.OptimizationsApi(base_api_client=client)

In [24]:
# %%time
# # Run optimizations
# for constraint_set in constraint_sets:
#     # Update task configuration
#     task_config['portfolio_constraints'] = constraint_set
#     task_config['risk_options'] = SDK.OptimizationTaskRiskOptions(
#         risk_model = SDK.RiskModelEnum.US_EQUITY,
#         #risk_model_scaling = SDK.RiskModelScalingEnum.WEEK,
#         risk_model_horizon = SDK.RiskModelHorizonEnum.ANNUAL
#     )
    
#     task_id = portfolio.create_task(**task_config)
#     result = portfolio.run_task(task_id, optimizer)
    
#     # Store results
#     result.wait_for_completion(optimization_api)
#     if result.wait_for_completion(optimization_api):
#         results_manager.store_optimization_result(
#             portfolio_id=portfolio.portfolio_id,
#             portfolio_name=portfolio.name,
#             task_name=task_config['name'],
#             task_id=task_id,
#             optimization_result=result,
#             constraint_set={
#                 c['fieldCode']: c['maxThreshold'] 
#                 for c in constraint_set
#             }
#         )

# results_df = results_manager.get_portfolio_results(portfolio.portfolio_id)

In [25]:
# mapper = {
#     "goal_CUSTOM_NUMBER(FIELD='CF_7459422178055815169')": "expected_return",
#     "constraint_active_total_risk": "actual_risk",
#     "constraint_turnover": "actual_turnover",
#     "constraint_maximum_positions": "actual_pos"
# }

# results_df

In [26]:
# commpleted_task_result.get_summary_dataframe()#.columns
# commpleted_task_result.get_goals_dataframe()
# commpleted_task_result.get_constraints_dataframe()
# #commpleted_task_result.get_trades_dataframe()
# commpleted_task_result.display_full_results()

In [27]:
# <h1 style="color:#9bddff">Portfolio Optimization & Order Staging Automation</h1>

In [ ]:
#frame = result.get_trades_dataframe()

In [ ]:
#test_frame = frame.head()
# test_frame = test_frame.rename(columns={'ticker':'security_id', 'changedQuantity':'quantity'})
# test_frame['side'] = test_frame['quantity'].apply(lambda x: Side.BUY if x > 0 else Side.SELL)
# test_frame = test_frame.loc[:,['security_id','quantity','side']]
# test_frame.to_dict()
#test_frame

In [ ]:
# for order in response_dict['individual_responses']:

#     cancel_xml = CancelOrderXMLHanlder.get_request_xml_string(
#         order_id=order.order_id,
#         flow_control_flag=FlowControlTag.ACTIVE_ORDER,
#         # free_text = 'Testing a note',
#         check_pretrade_compliance=CheckPretradeCompliance.YES,
#         compliance_override_text='Canceling order - just an API test'
#     )

#     tsadf_response = send_open_order_staging_xml(
#         msg_type='F',
#         xml_string=cancel_xml
#     )

#     response_xml = read_response(tsadf_response)
#     response_obj = CancelOrderXMLHanlder.get_response_from_xml(response_xml)
#     response_obj.to_dict()